# MentalBERT Multilabel Training Pipeline (Cleaned)

This notebook loads a labeled JSONL dataset, enforces a fixed 5-label schema, splits the data into train/validation/test sets, tokenizes the text, and prepares a BERT-style multilabel classifier for training. It also adds safer preprocessing, reproducible splits, and clearer sanity checks so the pipeline is easier to validate before full-scale training.

**Label order used everywhere:** `["Self-Harm", "Depression", "Aggression", "Anxiety", "Sexually suggestive"]`


## 1) Runtime check

In [ ]:
!nvidia-smi

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers

# Added datasets for memory-mapped loading
from datasets import load_dataset, DatasetDict
from sklearn.metrics import f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from transformers.modeling_outputs import SequenceClassifierOutput #
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

## 2) Central config

In [ ]:
# Central Config - Optimized for Memory Efficiency

MODEL_CKPT = "mental/mental-bert-base-uncased"
TEXT_COL = "text"
LABEL_COLS = ["Self-Harm", "Depression", "Aggression", "Anxiety", "Sexually suggestive"]

RANDOM_SEED = 123
TEST_SIZE = 0.10
VAL_SIZE = 0.10
MAX_LENGTH = 128
DEBUG_SAMPLE_SIZE = None

# Memory Optimized Training Settings
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2 # Effectively doubles batch size to 32 without VRAM cost
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
FP16 = True # Enables Mixed Precision for faster/lighter training

# DATA_DIR = Path("/content/gdrive/MyDrive")
DATA_PATH = "train.jsonl"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(device)

## 3) Mount Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/gdrive")

In [ ]:
from huggingface_hub import login
login()

## 4) Load and inspect the dataset

In [ ]:
# Using datasets.load_dataset to handle 1.5M rows via memory mapping (disk-based)
# This prevents RAM crashes.

raw_dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
df_raw = raw_dataset.to_pandas()


def preprocess_function(examples):
    # Create the multilabel 'labels' column
    labels_matrix = []
    for i in range(len(examples[TEXT_COL])):
        label_vec = [float(examples[col][i]) for col in LABEL_COLS]
        labels_matrix.append(label_vec)

    # Tokenize
    result = tokenizer(examples[TEXT_COL], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    result["labels"] = labels_matrix
    return result

# Shuffle and Split using Dataset.train_test_split
ds_split = raw_dataset.train_test_split(test_size=TEST_SIZE, seed=RANDOM_SEED)
train_val_ds = ds_split["train"].train_test_split(test_size=VAL_SIZE/(1-TEST_SIZE), seed=RANDOM_SEED)

final_ds = DatasetDict({
    "train": train_val_ds["train"],
    "validation": train_val_ds["test"],
    "test": ds_split["test"]
})

print("Dataset splits created successfully.")

In [ ]:
# Quick inspection cell
display(df_raw.head(3))
print("\nNull counts (top 10):")
display(df_raw.isnull().sum().sort_values(ascending=False).head(10))

## 5) Clean and enforce the label schema

In [ ]:
# Old notebook: used every non-text column as a label.
# New notebook: explicitly map the dataset into the 5 project labels in a fixed order.

def build_clean_dataframe(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    # Handle either nested-json column names (e.g., labels.Self-Harm)
    # or already-flattened label columns (e.g., Self-Harm).
    label_mapping = {}
    for label in LABEL_COLS:
        if label in df.columns:
            label_mapping[label] = label
        elif f"labels.{label}" in df.columns:
            label_mapping[label] = f"labels.{label}"
        else:
            raise ValueError(f"Could not find a column for label: {label}")

    # Keep only text + the 5 labels we care about.
    clean_df = pd.DataFrame()
    clean_df[TEXT_COL] = df[TEXT_COL]

    for label in LABEL_COLS:
        clean_df[label] = df[label_mapping[label]]

    # Drop rows with missing text.
    clean_df = clean_df.dropna(subset=[TEXT_COL]).copy()

    # Normalize text to strings and strip whitespace.
    clean_df[TEXT_COL] = clean_df[TEXT_COL].astype(str).str.strip()

    # Remove empty-text rows.
    clean_df = clean_df[clean_df[TEXT_COL] != ""].copy()

    # Convert labels to ints just in case they came in as floats/bools/strings.
    for label in LABEL_COLS:
        clean_df[label] = clean_df[label].fillna(0).astype(int)

    # Remove exact duplicate posts so they do not leak across splits.
    clean_df = clean_df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

    # Build the multilabel target vector in the fixed project order.
    clean_df["labels"] = clean_df[LABEL_COLS].values.tolist()

    return clean_df

df = build_clean_dataframe(df_raw)

# ------------------------------------------------------------------------------


if DEBUG_SAMPLE_SIZE is not None:
    df = df.sample(n=min(DEBUG_SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print("Using debug sample shape:", df.shape)
else:
    print("Using full cleaned dataset shape:", df.shape)

# ------------------------------------------------------------------------------

print("Clean shape:", df.shape)
display(df.head(3))
print(df[LABEL_COLS].sum())
print("Empty text rows:", (df["text"].astype(str).str.strip() == "").sum())

In [ ]:
# Sanity checks after cleaning
print("Label totals:")
display(df[LABEL_COLS].sum())

# Count neutral/all-zero posts because these matter for real-world balance.
neutral_mask = (df[LABEL_COLS].sum(axis=1) == 0)
print("Neutral / all-zero posts:", int(neutral_mask.sum()))
print("Non-neutral posts:", int((~neutral_mask).sum()))

In [ ]:
# Optional debug mode for faster experimentation.
# Old notebook: sampled 1.58M rows immediately.
# New notebook: sample only if you explicitly want a smaller dry-run dataset.

if DEBUG_SAMPLE_SIZE is not None:
    df = df.sample(n=min(DEBUG_SAMPLE_SIZE, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    print("Using debug sample shape:", df.shape)
else:
    print("Using full cleaned dataset shape:", df.shape)

## 6) Split into train / validation / test

In [ ]:
# Old notebook: used a single random 80/20 train-test mask.
# New notebook: create reproducible train/validation/test splits so test stays untouched until the end.

# First split off the final test set.
df_train_val, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True,
)

# Then split the remaining data into train and validation.
# This converts the desired 10% full-data validation split into the correct fraction of train_val.
val_relative_size = VAL_SIZE / (1.0 - TEST_SIZE)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=val_relative_size,
    random_state=RANDOM_SEED,
    shuffle=True,
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)

In [ ]:
# Check label balance across splits.
# This helps confirm that no important label became unusually rare in val/test.

def summarize_split(name: str, split_df: pd.DataFrame):
    summary = split_df[LABEL_COLS].mean().to_frame(name="positive_rate")
    summary["positive_count"] = split_df[LABEL_COLS].sum()
    summary["split"] = name
    return summary.reset_index().rename(columns={"index": "label"})

split_summary = pd.concat([
    summarize_split("train", df_train),
    summarize_split("val", df_val),
    summarize_split("test", df_test),
], ignore_index=True)

display(split_summary)

In [ ]:
print(len(df_train), len(df_val), len(df_test))

## 7) Tokenizer and encodings

In [ ]:
# This is Transformer API dataset pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

# Map the tokenization and labeling across the shards
tokenized_ds = final_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_dataset.column_names
)

train_dataset = tokenized_ds["train"]
val_dataset = tokenized_ds["validation"]
test_dataset = tokenized_ds["test"]

## 8) Dataset wrapper + sanity checks

In [ ]:
# This is ManualPytroch dataset pipeline

# Old notebook: class name was JigsawDataset (left over from a different project/tutorial).
# New notebook: rename it so the class matches this project.

# class MentalHealthDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
#         return item

#     def __len__(self):
#         return len(self.labels)
    

# # Convert the 5 label columns into a single list of lists (the target matrix)
# train_labels = df_train[LABEL_COLS].values.tolist()
# val_labels   = df_val[LABEL_COLS].values.tolist()
# test_labels  = df_test[LABEL_COLS].values.tolist()

# train_encodings = tokenizer(
#     list(df_train[TEXT_COL]), 
#     truncation=True, 
#     padding=True, 
#     max_length=MAX_LENGTH
# )

# val_encodings = tokenizer(
#     list(df_val[TEXT_COL]), 
#     truncation=True, 
#     padding=True, 
#     max_length=MAX_LENGTH
# )

# test_encodings = tokenizer(
#     list(df_test[TEXT_COL]), 
#     truncation=True, 
#     padding=True, 
#     max_length=MAX_LENGTH
# )


# train_dataset = MentalHealthDataset(train_encodings, train_labels)
# val_dataset = MentalHealthDataset(val_encodings, val_labels)
# test_dataset = MentalHealthDataset(test_encodings, test_labels)

In [ ]:
# Sanity-check one tokenized example before training.
sample_item = train_dataset[0]

print("Keys:", sample_item.keys())
print("Encoded label vector:", sample_item["labels"])
print("\nDecoded text preview:")
print(tokenizer.decode(sample_item["input_ids"], skip_special_tokens=True))

## 9) Define the multilabel BERT model

We use **BCEWithLogitsLoss** because each post can belong to multiple labels at once.  
That means this is **not** a single-choice softmax problem; each label is treated as an independent yes/no decision.


In [ ]:
# Old notebook: custom class was correct, but it appeared later after some inconsistent setup.
# New notebook: keep the custom model definition in a dedicated model section.

class BertForMultilabelSequenceClassification(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs[1]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            # BCEWithLogitsLoss = sigmoid + binary cross-entropy in one numerically stable step.
            loss_fct = torch.nn.BCEWithLogitsLoss()
            loss = loss_fct(
                logits.view(-1, self.num_labels),
                labels.float().view(-1, self.num_labels),
            )

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
# Old notebook: reloaded the tokenizer here and swapped classes in-place without much context.
# New notebook: load the base checkpoint once, set the label count, and then convert it to the custom multilabel class.

num_labels = len(LABEL_COLS)

base_model = BertForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    use_safetensors=True # use safetensor=True or upgrade pytorch because there is vulnability somewhere idk
)

# Change the loaded model to use the multilabel forward/loss logic above.
base_model.__class__ = BertForMultilabelSequenceClassification

model = base_model.to(device)
print("Model loaded on:", device)

## 10) Metrics

In [ ]:
# Old notebook: only computed thresholded accuracy.
# New notebook: report F1/precision/recall in a multilabel-friendly way.
# New notebook v2: add specific thresholds to each label (instead of one general threshhold)

from sklearn.metrics import precision_recall_fscore_support, f1_score
import numpy as np

# Change these later if needed
THRESHOLDS = {
    "Self-Harm": 0.6,
    "Depression": 0.5,
    "Aggression": 0.5,
    "Anxiety": 0.5,
    "Sexually suggestive": 0.3,
}

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert logits to probabilities
    probs = 1 / (1 + np.exp(-logits))

    # Apply a different threshold per label
    thresh_array = np.array([THRESHOLDS[label] for label in LABEL_COLS])
    preds = (probs >= thresh_array).astype(int)

    # Overall metrics
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    metrics = {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
    }

    # Per-label metrics
    per_label_precision, per_label_recall, per_label_f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    for i, label in enumerate(LABEL_COLS):
        metrics[f"{label}_precision"] = per_label_precision[i]
        metrics[f"{label}_recall"] = per_label_recall[i]
        metrics[f"{label}_f1"] = per_label_f1[i]

    return metrics

## 11) Training setup

In [ ]:
training_args = TrainingArguments(
    output_dir="mentalbert_multilabel_outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    fp16=FP16,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2 # Keep only 2 latest checkpoints to save Drive space
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # evaluate on validation, not test
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

## 12) Sanity-check evaluation, training, and final test

In [ ]:
# First sanity check: make sure evaluation runs before full training.
val_metrics = trainer.evaluate()
val_metrics

In [ ]:
# Then do the actual training run.
train_output = trainer.train()


# Save model + tokenizer
trainer.save_model("saved_model_output_AHHHHHHH")
tokenizer.save_pretrained("model_output_AHHHHHH")

# Predicted vs True Positives

In [ ]:
# pred_output = trainer.predict(val_dataset)

# logits = pred_output.predictions
# true_labels = pred_output.label_ids

# probs = 1 / (1 + np.exp(-logits))
# thresh_array = np.array([THRESHOLDS[label] for label in LABEL_COLS])
# preds = (probs >= thresh_array).astype(int)

# print("Total predicted positives per label:")
# print({label: int(preds[:, i].sum()) for i, label in enumerate(LABEL_COLS)})

# print("\nTotal true positives per label:")
# print({label: int(true_labels[:, i].sum()) for i, label in enumerate(LABEL_COLS)})


# # The difference in accuracy between the "Predicted Positives" and "True Positives"
# print("\nAccuracy per label (%):")
# for i, label in enumerate(LABEL_COLS):
#     label_accuracy = (preds[:, i] == true_labels[:, i]).mean() * 100
#     print(f"{label}: {label_accuracy:.2f}%")

# # Shows how far off the total predicted positive count is from the true count.
# # Good for spotting overprediction/underprediction, but not whether the exact rows were right.
# print("\nPositive-count error per label (%):")
# for i, label in enumerate(LABEL_COLS):
#     pred_pos = preds[:, i].sum()
#     true_pos = true_labels[:, i].sum()

#     if true_pos == 0:
#         error_pct = 0.0 if pred_pos == 0 else float("inf")
#     else:
#         error_pct = abs(pred_pos - true_pos) / true_pos * 100

#     if error_pct == float("inf"):
#         print(f"{label}: inf (true positives = 0)")
#     else:
#         print(f"{label}: {error_pct:.2f}%")

# Row by Row inspection

In [ ]:
# print("Train label totals:")
# print(df_train[LABEL_COLS].sum())

# print("\nVal label totals:")
# print(df_val[LABEL_COLS].sum())

# print("\nTest label totals:")
# print(df_test[LABEL_COLS].sum())

# Show Mistakes Cell

In [ ]:
# Rows where at least one label prediction differs from the true label

# Rebuild inspect_df here so this cell is self-contained
# inspect_df = df_val[[TEXT_COL] + LABEL_COLS].copy().reset_index(drop=True)

# for i, label in enumerate(LABEL_COLS):
#     inspect_df[f"pred_{label}"] = preds[:, i]
#     inspect_df[f"prob_{label}"] = probs[:, i]

# mismatch_mask = (preds != true_labels).any(axis=1)

# mistakes_df = inspect_df[mismatch_mask].copy()
# display(mistakes_df.head(30))
# print("Total mismatched validation rows:", mismatch_mask.sum())

# DO NOT RUN THE CELL (TEST_METRICS) BELOW UNTIL WE ARE DONE TRAINING, IT'S NOT READY YET.....

# Test Metrics

In [ ]:
# Final test evaluation should happen only after training/tuning is done.
# This is the "untouched until the end" split.

# test_metrics = trainer.evaluate(eval_dataset=test_dataset)
# test_metrics

# Training Results So Far


In [ ]:
# | Run | Sample | Ep | BS | Thresholds (SH / Dep / Agg / Anx / Sex) | Result |
# |-----|--------|----|----|------------------------------------------|--------|
# | 1   | 1000   | 1  | 8  | 0.5 / 0.5 / 0.5 / 0.5 / 0.5              | Collapsed to all-zero predictions; predicted = [0, 0, 0, 0, 0] |
# | 2   | 5000   | 3  | 8  | 0.3 / 0.3 / 0.3 / 0.3 / 0.3              | Overpredicted SH/Agg/Anx and underpredicted Sex; predicted = [209, 100, 281, 500, 22], true = [7, 47, 24, 40, 110] |
# | 3   | 5000   | 3  | 8  | 0.7 / 0.5 / 0.7 / 0.7 / 0.3              | First per-label tuning suggestion |
# | 4   | 5000   | 3  | 8  | 0.6 / 0.5 / 0.6 / 0.6 / 0.4              | Additional per-label tuning pass after global-threshold tests |
# | 5   | 5000   | 3  | 8  | 0.6 / 0.5 / 0.6 / 0.6 / 0.4              | Predicted = [1, 44, 13, 23, 98], true = [7, 47, 24, 40, 110]; underpredicted most labels |
# | 6   | 5000   | 3  | 8  | 0.6 / 0.5 / 0.5 / 0.5 / 0.3              | Suggested next tuning pass to loosen Agg/Anx/Sex while keeping SH moderate |
# | 7   | 5000   | 3  | 8  | 0.6 / 0.5 / 0.5 / 0.5 / 0.3              | Predicted = [6, 55, 21, 37, 97], true = [7, 47, 24, 40, 110]; Acc (%) = [98.60, 94.81, 96.21, 95.41, 92.22]; Pos-count err (%) = [14.29, 17.02, 12.50, 7.50, 11.82] |

# Next threshold pass to try later (wrote this at 04/20/2026):
# SH stays at 0.6 since 6 vs 7 was already close.
# Dep goes to 0.55 because it slightly overpredicted (55 vs 47).
# Agg and Anx go to 0.45 because they slightly underpredicted (21 vs 24, 37 vs 40).
# Sex goes to 0.25 because it was still underpredicting (97 vs 110).

# THRESHOLDS = {
#     "Self-Harm": 0.6,
#     "Depression": 0.55,
#     "Aggression": 0.45,
#     "Anxiety": 0.45,
#     "Sexually suggestive": 0.25,
# }

In [ ]:
# Experiment log
# Run 1: sample=1000, epochs=1, batch=8, thresholds=0.5/0.5/0.5/0.5/0.5
# Result: collapsed to all-zero predictions

# Run 2: sample=5000, epochs=3, batch=8, thresholds=0.3/0.3/0.3/0.3/0.3
# Result: predicted=[209,100,281,500,22], true=[7,47,24,40,110]
# Note: overpredicted Self-Harm/Aggression/Anxiety, underpredicted Sexually suggestive

# Run 3: sample=5000, epochs=3, batch=8, thresholds=0.7/0.5/0.7/0.7/0.3
# Result: first per-label tuning suggestion

# Run 4: sample=5000, epochs=3, batch=8, thresholds=0.6/0.5/0.6/0.6/0.4
# Result: additional per-label tuning pass added after global threshold tests

# Run 5: sample=5000, epochs=3, batch=8, thresholds=0.6/0.5/0.6/0.6/0.4
# Result: predicted=[1,44,13,23,98], true=[7,47,24,40,110]
# Note: much closer than Run 2, but now underpredicting most labels

# Run 6: sample=5000, epochs=3, batch=8, thresholds=0.6/0.5/0.5/0.5/0.3
# Result: next suggested tuning pass

## 13) Notes for later cleanup

- If you switch from `bert-base-uncased` to a real **MentalBERT** checkpoint later, you should only need to update `MODEL_CKPT`.
- If the full 1.5M dataset is too large for one Colab session, start with a smaller debug subset, confirm the pipeline works, then scale up.
- The next upgrade after this notebook is usually **per-label metrics** and **per-label threshold tuning**.
